In [1]:
"""
Applied test: instead of tuning lambda by cross-validation search, derive it
from an explicit Bayesian prior belief about the weights (the MAP
interpretation), then compare test performance against sklearn's RidgeCV
(which searches a grid via cross-validation).

lambda = sigma^2 / tau^2
  sigma^2 = noise variance, estimated from MLE residuals
  tau^2   = prior variance -> how large I genuinely believe standardized
            coefficients should be. For standardized features, a coefficient
            outside roughly [-1, 1] would be a surprisingly strong effect,
            so tau = 1 encodes "most true effects are of moderate size."
"""

import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.metrics import mean_squared_error, r2_score

from mle_map_regression import fit_mle, fit_map

data = load_diabetes()
X_raw, y = data.data, data.target

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42
)

scaler = StandardScaler().fit(X_train_raw)
X_train = np.column_stack([np.ones(len(X_train_raw)), scaler.transform(X_train_raw)])
X_test = np.column_stack([np.ones(len(X_test_raw)), scaler.transform(X_test_raw)])

# --- Step 1: estimate noise variance sigma^2 from MLE residuals ---
theta_mle = fit_mle(X_train, y_train)
resid = y_train - X_train @ theta_mle
n, p = X_train.shape
sigma2 = np.sum(resid ** 2) / (n - p)  # unbiased estimate

# --- Step 2: explicit prior belief -> tau ---
# First attempt: a naive, ungrounded belief that standardized coefficients
# should be order-1 (tau = 1). This is what "just pick a reasonable-sounding
# tau" looks like -- and it's checked against data below, not assumed correct.
tau_naive = 1.0
lambda_naive = sigma2 / (tau_naive ** 2)
theta_map_naive = fit_map(X_train, y_train, lambda_naive)
pred_naive = X_test @ theta_map_naive
mse_naive = mean_squared_error(y_test, pred_naive)
r2_naive = r2_score(y_test, pred_naive)

# Second, grounded belief: a standardized feature moving the outcome by up
# to one standard deviation of the target itself is a large but plausible
# effect. tau = std(y) anchors the prior to the actual scale of the problem
# instead of an arbitrary number.
tau = y_train.std()
tau2 = tau ** 2
lambda_bayesian = sigma2 / tau2

# --- Step 3: fit MAP (Ridge) with the derived lambda, evaluate on test set ---
theta_map = fit_map(X_train, y_train, lambda_bayesian)
pred_bayesian = X_test @ theta_map
mse_bayesian = mean_squared_error(y_test, pred_bayesian)
r2_bayesian = r2_score(y_test, pred_bayesian)

# --- Comparison: sklearn RidgeCV, which grid-searches alpha via CV ---
alphas = np.logspace(-2, 4, 100)
ridge_cv = RidgeCV(alphas=alphas, fit_intercept=True)
ridge_cv.fit(scaler.transform(X_train_raw), y_train)
pred_cv = ridge_cv.predict(scaler.transform(X_test_raw))
mse_cv = mean_squared_error(y_test, pred_cv)
r2_cv = r2_score(y_test, pred_cv)

# --- Also show plain MLE (lambda=0) for reference ---
pred_mle = X_test @ theta_mle
mse_mle = mean_squared_error(y_test, pred_mle)
r2_mle = r2_score(y_test, pred_mle)

print(f"Estimated noise variance (sigma^2) from MLE residuals: {sigma2:.2f}")
print(f"Naive prior:    tau=1.0            -> lambda={lambda_naive:.2f}")
print(f"Grounded prior: tau=std(y)={tau:.2f} -> lambda={lambda_bayesian:.2f}")
print(f"RidgeCV's cross-validated best alpha:                    {ridge_cv.alpha_:.2f}")
print()
print(f"{'Method':<30}{'Test MSE':>12}{'Test R2':>10}")
print("-" * 52)
print(f"{'MLE (lambda=0)':<30}{mse_mle:12.2f}{r2_mle:10.4f}")
print(f"{'MAP, naive tau=1':<30}{mse_naive:12.2f}{r2_naive:10.4f}")
print(f"{'MAP, grounded tau=std(y)':<30}{mse_bayesian:12.2f}{r2_bayesian:10.4f}")
print(f"{'Ridge, CV-searched alpha':<30}{mse_cv:12.2f}{r2_cv:10.4f}")


Estimated noise variance (sigma^2) from MLE residuals: 2960.81
Naive prior:    tau=1.0            -> lambda=2960.81
Grounded prior: tau=std(y)=77.95 -> lambda=0.49
RidgeCV's cross-validated best alpha:                    1.15

Method                            Test MSE   Test R2
----------------------------------------------------
MLE (lambda=0)                     2900.19    0.4526
MAP, naive tau=1                   4180.03    0.2110
MAP, grounded tau=std(y)           2895.44    0.4535
Ridge, CV-searched alpha           2891.21    0.4543
